# 02. 재무제표 받기 (파일럿)

**이 노트북이 하는 일:** 기업 40곳을 **어떻게 뽑았는지** 재현하고, 그 기업들의 재무제표를 DART에서 **어떻게 받는지**(받는 코드의 구조)를 설명하고, 이미 받아 둔 결과를 읽어서 **무엇이 왔고 무엇이 비었는지** 확인합니다.

| 항목 | 내용 |
|---|---|
| 입력 | `outputs/01_파일럿/기업개황_무작위150곳.csv`, `파일럿_표본40곳.csv`, `재무제표_수집로그.csv`, `data/raw/DART_파일럿/fs/*.json`(1,768개), `기업개황_상장사전체.csv` |
| 출력 | 화면 요약만 (기존 파일을 덮어쓰지 않음) |
| DART API 호출 | **기본값은 호출하지 않음**(`RUN_COLLECTION = False`). 저장된 파일만 읽음 |
| 실행 시간 | 약 5초 |
| 먼저 알아야 할 것 | 00번(상태코드·캐시), 01번(법인 명단) |
| 원래 스크립트 | `원본_스크립트/02_재무제표_수집_원본.py` |

## 이 노트북의 지도
1. "한 번의 요청"이 가져오는 것 — 파라미터 네 개
2. 파일럿 40곳은 어떻게 뽑혔나 (재현 + 대표성 문제)
3. 호출 수 계산
4. 받는 코드의 구조 — 연결 우선, 없으면 별도, 캐시, 로그 (**실제 수집은 스위치로 막아 둠**)
5. 저장된 응답 한 건 열어 보기
6. 로그로 본 결과 — 자료가 있는 기업과 없는 기업
7. 확인하지 못한 것

> ### ⚠ 안전장치
> `RUN_COLLECTION`을 `True`로 바꾸고 인증키가 설정되어 있으면 **저장되지 않은 보고서를 DART에 요청**합니다. 이 노트북은 "전체 실행(Run All)"을 해도 기본값(`False`)에서는 DART에 아무 요청도 보내지 않습니다. **본 수집(수천 곳)은 이 노트북이 아니라 앞으로 만들 별도 스크립트로 합니다.**

In [1]:
import csv, json, random, re, sys, time
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import config
from dart_api import CACHEABLE_STATUSES, FatalApiStatus, LimitExceeded, get_json

RUN_COLLECTION = False     # ← True로 바꾸면 저장 안 된 보고서를 DART에 요청함. 기본은 False

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 120)
print("DART 요청 허용:", RUN_COLLECTION)

DART 요청 허용: False


---
# 1. "한 번의 요청"이 가져오는 것

`fnlttSinglAcntAll`(단일회사 전체 재무제표) 요청은 **네 가지를 지정**합니다.

| 파라미터 | 뜻 | 우리 값 |
|---|---|---|
| `corp_code` | 어느 회사 | 8자리 번호 |
| `bsns_year` | 사업연도 | 2018 ~ 2024 (7개) |
| `reprt_code` | 어느 보고서 | 11013 1분기 · 11012 반기 · 11014 3분기 · 11011 사업(연간) |
| `fs_div` | 연결/별도 | CFS 연결 · OFS 별도 |

- **한 번의 요청 = 한 회사의 한 보고서 안의 재무제표 전체**(계정 100~수백 줄)입니다.
- **4분기 보고서는 없습니다.** 그래서 4분기 값은 나중에 "연간 − 3분기 누적"으로 계산합니다(04번 노트북).

한 기업당 조합은 7년 × 4보고서 = **28개**이고, 각 조합에서 연결부터 묻고 비어 있으면 별도를 묻습니다.

---
# 2. 파일럿 40곳은 어떻게 뽑혔나

기억할 점: **결과(ICR 등)를 보지 않고** 뽑았습니다. 결과를 보고 고르면 "예쁜 결과가 나오는 기업만" 남기는 선택 편향이 생깁니다.

과정은 세 단계였고, 아래에서 **같은 번호(SEED)로 똑같이 재현**해서 원래 파일과 같은지 확인합니다.

1. 상장사 4,009곳을 섞어서(시드 고정) 앞 150곳을 뽑아 기업개황을 받음 → `기업개황_무작위150곳.csv` (업종코드 체계를 눈으로 보려는 용도)
2. 150곳 중 **제조업**(업종 앞 2자리 10~34)만 남김
3. **층화 라운드로빈**으로 40곳 추출 → `파일럿_표본40곳.csv`

In [2]:
# 1단계 재현: 01번 노트북과 같은 순서의 상장사 목록 → 섞기 → 앞 150곳
import io, zipfile, xml.etree.ElementTree as ET

with zipfile.ZipFile(config.RAW / "corpCode.zip") as zf:
    root = ET.fromstring(zf.read(zf.namelist()[0]))
listed = []
for node in root.findall("list"):
    stock = (node.findtext("stock_code") or "").strip()
    if stock:
        listed.append((node.findtext("corp_code").strip(), node.findtext("corp_name").strip()))

targets = list(listed)
random.Random(config.SEED).shuffle(targets)
probe_codes = [c for c, _ in targets[: config.N_COMPANY_PROBE]]

probe = pd.read_csv(config.OUT / "기업개황_무작위150곳.csv", dtype=str, keep_default_na=False)
print("재현한 150곳이 기업개황_무작위150곳.csv와 같은 기업인가:", set(probe_codes) == set(probe["corp_code"]))
print("probe 법인구분:", dict(Counter(probe["corp_cls"])))

재현한 150곳이 기업개황_무작위150곳.csv와 같은 기업인가: True
probe 법인구분: {'Y': 32, 'E': 41, 'K': 70, 'N': 7}


### 해석
같은 시드로 섞으면 같은 150곳이 나옵니다. **시드가 "재현 가능한 무작위"** 의 핵심입니다. 이것이 틀리면 원래 표본을 우리가 다시 만들 수 없다는 뜻이라 확인했습니다.

150곳의 법인구분에서 **E(기타)가 약 27%** (41곳)로, 실제 상장사 구성(코스피·코스닥 위주)보다 많이 보입니다. 이전 문서에서 "현재 시장구분으로 거르지 말자"고 한 근거입니다.

In [3]:
# 2~3단계 재현: 제조업만 남기고, (법인구분, 업종 앞 2자리)별로 묶어 돌아가며 한 곳씩 뽑는다
def is_manufacturing(code, prefixes):
    return any(code.startswith(p) for p in prefixes)

def stratified_sample(rows, n, seed):
    groups = defaultdict(list)
    for r in rows:
        groups[(r["corp_cls"], r["induty_code"][:2])].append(r)      # 같은 (법인구분, 업종2자리)끼리 한 묶음(층)
    rng = random.Random(seed)
    keys = sorted(groups)
    for k in keys:
        rng.shuffle(groups[k])                                       # 각 묶음 안에서 섞기
    picked = []
    while len(picked) < n and any(groups[k] for k in keys):
        for k in keys:                                               # 묶음을 한 바퀴 돌며 한 곳씩
            if groups[k] and len(picked) < n:
                picked.append(groups[k].pop())
    return picked

probe_rows = probe.to_dict("records")
manu = [r for r in probe_rows if is_manufacturing(r["induty_code"], config.MANUFACTURING_PREFIXES)]
sample = stratified_sample(manu, config.N_PILOT, config.SEED)

saved = pd.read_csv(config.OUT / "파일럿_표본40곳.csv", dtype=str, keep_default_na=False)
print(f"probe 150곳 중 제조업: {len(manu)}곳")
print(f"뽑은 표본: {len(sample)}곳")
print("원래 파일럿_표본40곳.csv와 같은 기업·같은 순서인가:", [r['corp_code'] for r in sample] == list(saved['corp_code']))

probe 150곳 중 제조업: 84곳
뽑은 표본: 40곳
원래 파일럿_표본40곳.csv와 같은 기업·같은 순서인가: True


### 해석
- 제조업만 남기면 **84곳**이고, 그중 40곳을 뽑았습니다. 재현한 결과가 원래 파일과 **같은 기업, 같은 순서**입니다.
- **층화 라운드로빈**이란 이름 그대로입니다. 비슷한 기업끼리 묶음(층)을 만들고, **묶음을 돌아가며 한 곳씩** 뽑습니다. 목적은 "특정 업종·시장에 쏠리지 않고 다양하게 보기"입니다.

그런데 이 방식에는 **구조적 문제**가 있습니다. 아래에서 확인합니다.

In [4]:
cmp = pd.DataFrame({
    "제조업 84곳(후보)": pd.Series(Counter(r["corp_cls"] for r in manu)),
    "뽑힌 40곳": pd.Series(Counter(r["corp_cls"] for r in sample)),
}).fillna(0).astype(int)
cmp["후보 중 비중"] = (cmp["제조업 84곳(후보)"] / cmp["제조업 84곳(후보)"].sum()).round(3)
cmp["표본 중 비중"] = (cmp["뽑힌 40곳"] / cmp["뽑힌 40곳"].sum()).round(3)
cmp.index.name = "법인구분"
display(cmp)

strata = Counter((r["corp_cls"], r["induty_code"][:2]) for r in manu)
print(f"층(묶음) 개수: {len(strata)}개, 한 층에 기업이 1곳뿐인 층: {sum(1 for v in strata.values() if v == 1)}개")

,제조업 84곳(후보),뽑힌 40곳,후보 중 비중,표본 중 비중
법인구분,,,,
E,14,13,0.167,0.325
K,46,14,0.548,0.350
N,3,3,0.036,0.075
Y,21,10,0.250,0.250


층(묶음) 개수: 37개, 한 층에 기업이 1곳뿐인 층: 22개


### 해석
- 층이 **37개**이고 그중 **22개는 기업이 1곳뿐**입니다. 라운드로빈은 층마다 한 곳씩 뽑기 때문에, **기업이 많은 큰 층과 기업이 1곳뿐인 작은 층이 같은 기회**를 받습니다.
- 그 결과 표본의 비중이 후보의 비중과 달라집니다(위 표). **법인구분 E는 후보의 16.7%(14곳)인데 표본에서는 32.5%(13곳)** 이고, 코스닥(K)은 후보 54.8%에서 표본 35.0%로 줄었습니다. **이 40곳은 모집단의 축소판이 아닙니다.** 파일럿의 목적은 "자료가 추출되는지"를 보는 것이었고 대표성은 목표가 아니었습니다. 40이라는 숫자도 설정 파일의 임의 값입니다.
- 앞으로 본 수집은 **선별을 통과한 기업을 무작위 순서로 줄 세워 앞에서부터 받는** 방식으로 바꾸자는 제안이 있습니다(아직 미작성).

---
# 3. 호출 수 계산

In [5]:
n_firms = len(sample)
n_years = len(config.YEARS)
n_reports = len(config.REPORTS)
combos = n_firms * n_years * n_reports
print(f"기업 {n_firms} × 연도 {n_years} × 보고서 {n_reports} = {combos:,} 조합")
print(f"연결이 비어서 별도로 다시 묻는 경우까지 최대: {combos * len(config.FS_ORDER):,} 건")
print()
print("본 수집 가정 (이전 메모의 추정치):")
print(f"  기업당 약 39호출 × 제조업 2,250곳 ≈ {39*2250:,} 호출")
print(f"  호출 사이 대기 {config.SLEEP}초 → 순수 대기 시간만 약 {39*2250*config.SLEEP/3600:.1f} 시간")

기업 40 × 연도 7 × 보고서 4 = 1,120 조합
연결이 비어서 별도로 다시 묻는 경우까지 최대: 2,240 건

본 수집 가정 (이전 메모의 추정치):
  기업당 약 39호출 × 제조업 2,250곳 ≈ 87,750 호출
  호출 사이 대기 0.4초 → 순수 대기 시간만 약 9.8 시간


### 해석
- 파일럿 조합은 1,120개이고, 실제 호출은 **1,768건**이었습니다(아래 6절). 연결에 자료가 없으면 별도를 한 번 더 묻기 때문에 조합 수보다 호출이 많습니다.
- **전수 수집은 약 88,000건**으로 추정되는데, **일일 요청 한도를 모릅니다.** 가이드에 "일반적으로 20,000건 이상"이라고만 있고 기간 단위가 불분명합니다. 한도가 20,000건/일이라고 가정하면 4~5일이 걸립니다. 이 추정은 확인되지 않은 가정 위에 있습니다.
- 대기 시간은 서버가 응답하는 시간을 포함하지 않은 최소치입니다.

---
# 4. 받는 코드의 구조

아래 두 함수가 원본_스크립트/02_재무제표_수집_원본.py의 핵심입니다. **실제 수집은 `RUN_COLLECTION=True`일 때만** 일어납니다.

### 4-1. 보고서 한 건 가져오기 `fetch_report()`
```
저장된 파일 있나? ── 있음 → 그 파일 읽기 (요청 안 함)
        └ 없음 → 요청 허용 안 됨 → '없음'으로 돌려줌
               → 요청 허용 → DART에 요청 → 000/013이면 파일로 저장 → 0.4초 쉼
```
저장 파일 이름은 `{corp_code}_{연도}_{보고서코드}_{CFS|OFS}.json`이라서, **파일 이름만 봐도 어떤 질문의 답인지** 압니다.

In [6]:
FS_DIR = config.RAW / "fs"

def fetch_report(corp, year, rep_code, fs, allow_api):
    # 보고서 한 건을 돌려준다. (응답, 출처) 출처: 'cache' / 'api' / 'missing'
    path = FS_DIR / f"{corp}_{year}_{rep_code}_{fs}.json"
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8")), "cache"
    if not allow_api:
        return None, "missing"
    data = get_json("fnlttSinglAcntAll.json",
                    {"corp_code": corp, "bsns_year": str(year), "reprt_code": rep_code, "fs_div": fs})
    if data.get("status") in CACHEABLE_STATUSES:       # 000, 013만 저장
        path.parent.mkdir(exist_ok=True)
        path.write_text(json.dumps(data, ensure_ascii=False), encoding="utf-8")
    time.sleep(config.SLEEP)
    return data, "api"

### 4-2. 기업 여러 곳을 돌기 `collect()` — 연결 우선, 없으면 별도
```
기업 → 연도 → 보고서(Q1,H1,Q3,FY) → 연결(CFS) 요청
                                      ├ 000 (자료 있음) → 기록하고 다음 보고서로 (별도는 묻지 않음)
                                      ├ 013 (자료 없음) → 기록하고 별도(OFS)를 다시 묻기
                                      └ 그 외 → 메시지 출력하고 이 보고서는 중단
한도(020)나 치명 오류 → 반복 전체 중단, 지금까지의 기록(로그)은 파일로 저장
```
한 기업 안에서 "어떤 해는 연결, 어떤 해는 별도"가 섞일 수 있습니다(6절에서 확인). 이 규칙 자체가 그런 혼재를 만듭니다.

In [7]:
def collect(firms, allow_api):
    log = []
    try:
        for firm in firms:
            for year in config.YEARS:
                for rep_name, rep_code in config.REPORTS.items():
                    for fs in config.FS_ORDER:                        # ['CFS', 'OFS'] 연결부터
                        data, src = fetch_report(firm["corp_code"], year, rep_code, fs, allow_api)
                        if data is None:                              # 저장도 없고 요청도 못 하는 경우
                            break
                        status = data.get("status")
                        log.append({
                            "corp_code": firm["corp_code"], "year": year, "report": rep_name, "fs_div": fs,
                            "status": status,
                            "n_rows": len(data.get("list", [])) if status == "000" else 0,
                            "cached": src == "cache",
                        })
                        if status == "000":
                            break                                     # 연결에 자료가 있으면 별도는 묻지 않음
                        if status != "013":
                            print(f"  {firm['corp_code']} {year} {rep_name} {fs}: 상태 {status} {data.get('message')}")
                            break
    except LimitExceeded as exc:
        print(str(exc))
    except FatalApiStatus as exc:
        print("중단:", exc)
    return pd.DataFrame(log)


if RUN_COLLECTION:
    print("⚠ 실제 수집을 시작합니다 (저장 안 된 것만 요청)")
else:
    print("RUN_COLLECTION=False → 저장된 파일만 읽습니다 (DART 요청 없음)")

# 저장된 파일만으로 로그를 다시 만들어 본다
log = collect(sample, allow_api=RUN_COLLECTION)
print("재구성한 로그 행 수:", len(log))
log.head()

RUN_COLLECTION=False → 저장된 파일만 읽습니다 (DART 요청 없음)


재구성한 로그 행 수: 1768


,corp_code,year,report,fs_div,status,n_rows,cached
0,00103325,2018,Q1,CFS,013,0,True
1,00103325,2018,Q1,OFS,013,0,True
2,00103325,2018,H1,CFS,013,0,True
3,00103325,2018,H1,OFS,013,0,True
4,00103325,2018,Q3,CFS,013,0,True


### 해석 — 이 코드가 원래 로그를 정확히 재현하는가?
원래 수집 때 남은 `재무제표_수집로그.csv`와 **같은 행들**을 이 노트북이 만들어 내는지 비교합니다. 같다면 "이 노트북의 `collect()`가 원래 스크립트와 같은 규칙으로 동작한다"는 뜻입니다. (`cached` 칸은 당시엔 False였으므로 비교에서 뺍니다.)

In [8]:
orig = pd.read_csv(config.OUT / "재무제표_수집로그.csv", dtype={"corp_code": str, "status": str})
cols = ["corp_code", "year", "report", "fs_div", "status", "n_rows"]
a = orig[cols].reset_index(drop=True)
b = log[cols].reset_index(drop=True)
print("행 수 같음:", len(a) == len(b), f"(원래 {len(a):,} / 재구성 {len(b):,})")
print("내용 모두 같음:", a.equals(b))

행 수 같음: True (원래 1,768 / 재구성 1,768)
내용 모두 같음: True


`True / True`면 규칙이 원래와 같습니다. 이는 **코드가 같다는 것을 보인 것**이지, **자료가 옳다는 것**을 보인 것은 아닙니다.

---
# 5. 저장된 응답 한 건 열어 보기

디와이덕양(`00114154`)의 2022년 3분기 보고서, 연결입니다.

In [9]:
resp = json.loads((FS_DIR / "00114154_2022_11014_CFS.json").read_text(encoding="utf-8"))
print("상태:", resp["status"], resp["message"], "| 계정 줄 수:", len(resp["list"]))
print("한 줄의 칸 이름:", list(resp["list"][0].keys()))
print()
df = pd.DataFrame(resp["list"])
print("어느 표(sj_div)에 몇 줄:")
print(df["sj_div"].value_counts().to_string())

상태: 000 정상 | 계정 줄 수: 111
한 줄의 칸 이름: ['rcept_no', 'reprt_code', 'bsns_year', 'corp_code', 'sj_div', 'sj_nm', 'account_id', 'account_nm', 'account_detail', 'thstrm_nm', 'thstrm_amount', 'frmtrm_nm', 'frmtrm_amount', 'ord', 'currency']

어느 표(sj_div)에 몇 줄:
sj_div
BS     41
SCE    25
CF     23
CIS    22


In [10]:
show = ["sj_div", "account_id", "account_nm", "thstrm_nm", "thstrm_amount", "thstrm_add_amount", "currency"]
df[df["account_nm"].isin(["자산총계", "단기차입금", "매출액", "영업이익(손실)", "금융비용", "이자지급"])][show]

,sj_div,account_id,account_nm,thstrm_nm,thstrm_amount,thstrm_add_amount,currency
17,BS,ifrs-full_Assets,자산총계,제 46 기 3분기말,527622764099,NaN,KRW
21,BS,ifrs-full_ShorttermBorrowings,단기차입금,제 46 기 3분기말,32630060000,NaN,KRW
41,CIS,ifrs-full_Revenue,매출액,제 46 기 3분기,388972308578,1090581684377,KRW
45,CIS,dart_OperatingIncomeLoss,영업이익(손실),제 46 기 3분기,4249354059,8288778136,KRW
49,CIS,ifrs-full_FinanceCosts,금융비용,제 46 기 3분기,923864683,2624345691,KRW
66,CF,ifrs-full_InterestPaidClassifiedAsOperatingAct...,이자지급,제 46 기 3분기,2622718882,NaN,KRW


### 해석
| 칸 | 뜻 |
|---|---|
| `sj_div` | 어느 표인가. BS 재무상태표 / CIS 포괄손익계산서 / CF 현금흐름표 / SCE 자본변동표 |
| `account_id` | 표준 계정코드. `ifrs-full_Assets`처럼 생긴 것은 표준, **회사가 자기 식으로 만든 계정은 "표준계정코드 미사용"으로 옴** |
| `account_nm` | 회사가 적은 이름. **회사마다 다를 수 있음** |
| `thstrm_amount` | 당기 금액. 손익은 **그 분기 3개월** 값(가이드), 재무상태표는 그 날짜의 값 |
| `thstrm_add_amount` | 당기 **누적** 금액 (손익 계정에서 1월부터 그 분기말까지). 재무상태표엔 없음(비어 있음) |
| `currency` | 통화. 위에선 KRW(원) |

- 위 3분기 보고서의 영업이익은 `thstrm_amount`(7~9월 3개월) **4,249,354,059**, `thstrm_add_amount`(1~9월 누적) **8,288,778,136**입니다. 두 숫자는 다른 것이므로 **섞어 쓰면 안 됩니다.**
- 이 기업은 **이자비용이 아니라 '금융비용'** 이름의 계정이 있습니다. 파일럿 전체에서 이런 경우가 대부분이었습니다(03번 노트북).
- 현금흐름표의 **'이자지급'은 `thstrm_add_amount`가 비어 있고** `thstrm_amount`에 누적 값이 들어 있는 것으로 보입니다(1분기 756.8M → 반기 1,702.6M…). 가이드에 현금흐름표 분기 규정 설명이 없어서 "관찰로 그렇게 보임" 수준입니다.

---
# 6. 로그로 본 결과 — 무엇이 왔고 무엇이 비었나

In [11]:
print("호출 결과 분포 (재무제표 구분 × 상태):")
print(pd.crosstab(log["fs_div"], log["status"]).to_string())
print()
print(f"전체 호출 {len(log):,}건")
ok = log[log["status"] == "000"]
print(f"자료가 온 보고서(연결+별도): {len(ok)}개")

호출 결과 분포 (재무제표 구분 × 상태):
status  000  013
fs_div          
CFS     472  648
OFS     179  469

전체 호출 1,768건
자료가 온 보고서(연결+별도): 651개


### 해석
- 연결 요청 1,120건 중 **자료가 온 것 472건**, 나머지 648건은 013(없음)이었습니다. 013이었던 건에서 별도를 다시 물어 **179건**이 자료가 왔고, 469건은 연결도 별도도 없었습니다.
- 그래서 총 호출 **1,768 = 472 + 648 + 179 + 469**, 자료가 있는 보고서는 **472 + 179 = 651개**입니다.
- **연결 자료가 없다는 것**은 이 기업에 자회사가 없다는 뜻일 수도(그래서 별도만 제출), 그 해 보고서 자체가 없다는 뜻일 수도 있습니다. 두 경우를 이 로그만으로는 구분하지 못합니다.

In [12]:
by_firm = log.assign(has=(log["status"] == "000")).groupby("corp_code")["has"].sum()
empty_codes = list(by_firm[by_firm == 0].index)
have_codes = list(by_firm[by_firm > 0].index)
print(f"표본 {n_firms}곳 중 자료가 한 번이라도 온 기업: {len(have_codes)}곳 / 전혀 없는 기업: {len(empty_codes)}곳")
print()
meta = pd.DataFrame(sample).set_index("corp_code")
print("자료가 전혀 없었던 기업:")
meta.loc[empty_codes, ["corp_name", "corp_cls", "induty_code", "acc_mt"]]

표본 40곳 중 자료가 한 번이라도 온 기업: 28곳 / 전혀 없는 기업: 12곳

자료가 전혀 없었던 기업:


,corp_name,corp_cls,induty_code,acc_mt
corp_code,,,,
00103325,고제,E,10895,12
00105509,금강화섬,E,13103,09
00158839,제너비오믹스,E,27214,12
00183914,우경,E,2419,12
00198572,동양매직,E,28519,12
00208301,성우몰드,E,29293,12
00349468,블루젬디앤씨,E,204,12
00677486,그린광학,K,273,12
01010572,패션플랫폼,E,14112,12


### 해석
**원인을 확인하지 못했습니다.** 가능한 설명(모두 가설)은 이렇습니다.
- 우리가 요청한 연도(2018~2024) 동안 **정기보고서를 제출한 적이 없는 법인**(예: 상장 전·후, 폐지, 코넥스·스팩 등)
- DART의 이 API가 **그 기업의 보고서를 제공하지 않는 경우**(예: 소규모, 형식 차이)
- 우리가 요청한 방식의 문제

위 표에서 보면 비어 있는 12곳은 **법인구분 E 8곳, 코스닥(K) 2곳, 코넥스(N) 2곳**이고 코스피(Y)는 없습니다. E가 많다는 점은 "폐지·스팩 등이 섞여 있다"는 앞선 추정과 방향이 맞지만, **"왜"는 알 수 없습니다.** 예를 들어 코스닥 2곳 중 하나는 최근에 상장해서 2018~2024년 보고서가 없을 수도 있으나 이는 추측이며 확인하지 않았습니다. 이 기업들을 표본에서 어떻게 다룰지(제외? 사유 기록?)가 사전 고정 문서의 "이탈 기업" 규칙(A7)과 연결됩니다. 특히 **사전기간에 상장했다가 폐지된 기업이 이 그룹에 포함돼 있다면 생존편향의 증거**일 수 있어서 반드시 사유를 확인해야 합니다(KIND 대조 필요).

In [13]:
# 기업 × 연도별로 '자료가 온 보고서 수(0~4)'를 본다 (자료가 있는 기업만)
got = log[log["status"] == "000"].groupby(["corp_code", "year"])["report"].nunique().unstack(fill_value=0)
got = got.reindex(have_codes).fillna(0).astype(int)
got.insert(0, "기업명", meta.loc[have_codes, "corp_name"].values)
got.insert(1, "구분", meta.loc[have_codes, "corp_cls"].values)
got

year,기업명,구분,2018,2019,2020,2021,2022,2023,2024
corp_code,,,,,,,,,
00109587,대륙제관,K,4,4,4,4,4,4,4
00114154,디와이덕양,Y,4,4,4,4,4,4,4
00117230,TYM,Y,4,4,4,4,4,4,4
00125822,삼보산업,K,4,4,4,4,4,4,4
00129271,삼화전기,Y,4,4,4,4,4,4,4
00136776,차AI헬스케어,Y,4,4,4,4,4,4,4
00140566,한탑,K,4,4,4,4,4,4,4
00140779,영보화학,Y,4,4,4,4,4,4,4
00152385,에이프로젠,Y,4,4,4,4,4,4,4


### 해석
- 숫자는 그 해에 **받은 보고서 수(0~4)** 입니다. 4면 Q1·반기·Q3·사업보고서가 모두 있는 해입니다.
- **0이나 1~3인 해가 보이는 기업**은 그 해 분기 값을 만들 수 없습니다(예: 4분기 = 연간 − 3분기 누적이므로 사업보고서와 3분기 보고서가 **둘 다** 있어야 함). 이런 해는 TTM(후행 4분기 합)이 비게 됩니다.
- 기업마다 자료가 있는 기간이 다릅니다. 이것이 **패널이 "불균형(unbalanced)"** 이라는 뜻입니다. 사전기간(2019Q1~2021Q2)에 모든 분기가 있는 기업만 쓸지(선별 규칙) 결정해야 하는 이유입니다.
- **자료가 늦게 시작하는 기업이 8곳** 보입니다(아주스틸, 에스앤디, 마녀공장, 에스엠씨지, 세니젠, 피에스케이, 미쥬, 와이앤넥스트는 일부 해가 0이거나 2~3). 이런 기업은 **사전기간(2019Q1~2021Q2)이 비어서** 분석에 못 씁니다. 늦게 상장했을 가능성이 있지만 확인하지 않았습니다.
- 이 표는 **파일럿 표본이 작아서 대표성이 없으므로** 비율을 일반화하면 안 됩니다.

### 제안된 선별 규칙을 파일럿 28곳에 적용해 보기
앞으로 본 수집에서 쓰자고 제안된 선별 규칙은 **"2019 사업보고서와 2021 반기보고서가 있는 기업만 남긴다"** (결과를 보지 않는 기준)입니다. 이를 파일럿 28곳에 적용하면 몇 곳이 남는지 세어 봅니다. 규칙을 **정하기 전에 효과를 가늠해 보는** 계산이며 아직 확정된 규칙이 아닙니다.

In [14]:
rep_has = log[log["status"] == "000"].groupby("corp_code").apply(
    lambda g: set(zip(g["year"], g["report"])), include_groups=False)

def passes(codeset):
    return (2019, "FY") in codeset and (2021, "H1") in codeset

passed = [c for c in have_codes if passes(rep_has[c])]
failed = [c for c in have_codes if not passes(rep_has[c])]
print(f"자료가 있던 {len(have_codes)}곳 중 통과: {len(passed)}곳 / 탈락: {len(failed)}곳")
print("탈락한 기업:", [meta.loc[c, 'corp_name'] for c in failed])
print(f"표본 40곳 기준으로는 {len(passed)}곳 통과 (자료가 전혀 없던 {len(empty_codes)}곳도 탈락)")

자료가 있던 28곳 중 통과: 22곳 / 탈락: 6곳
탈락한 기업: ['아주스틸', '에스앤디', '마녀공장', '에스엠씨지', '세니젠', '미쥬']
표본 40곳 기준으로는 22곳 통과 (자료가 전혀 없던 12곳도 탈락)


### 해석
- 이 규칙을 적용하면 파일럿 표본이 **40곳 → 22곳(55%)** 으로 줄어듭니다(자료가 전혀 없던 12곳 + 늦게 시작한 6곳 탈락). 즉 "명단 2,250곳 = 분석 표본"이 아니라 **선별 후에는 상당히 줄어들 수 있다**는 뜻입니다. 파일럿 40곳의 통과율을 전체에 그대로 적용하면 안 되지만(대표성이 없음), 최종 표본 크기를 가늠할 단서는 됩니다.
- 이 선별은 **ICR 같은 결과를 보지 않고** 보고서 존재 여부만 보므로 결과 선택 편향을 만들지 않습니다. 대신 **사전기간 이후에 사라진 기업(폐지 등)은 통과할 수 있다**는 점은 의도한 설계입니다(사전기간 기준 정의, 이탈은 기록).
- **이 규칙의 허점:** 피에스케이는 2019년 보고서가 3개뿐인데도(위 표) 규칙을 통과했습니다. 두 보고서의 **존재만** 보기 때문에, 사전기간에 분기 하나가 비어 있어도 통과할 수 있습니다. TTM(후행 4분기 합)이 비는 기간이 생기므로, 규칙을 더 엄격히 할지는 별도 결정이 필요합니다.
- 규칙의 두 보고서(2019 사업, 2021 반기)를 고른 이유는 "사전기간의 처음과 끝을 대표"하기 위한 **제안자의 선택이며 공식 기준이 아닙니다.**

---
# 7. 정리

### 이 노트북으로 확인된 것
- 파일럿 표본 40곳의 추출 과정을 같은 시드로 재현했고 원래 파일과 같음을 확인했습니다.
- 수집 코드의 규칙(연결 우선·별도 대체·000/013만 저장)을 구현하고, 저장된 파일만 읽어서 **원래 호출 로그를 재현**할 수 있음을 확인했습니다.
- DART는 새로 호출하지 않았습니다.

### 확인하지 못한 것 (문제점)
| 문제 | 이유 | 관련 결정 |
|---|---|---|
| 자료가 전혀 없는 기업의 사유 | 이 로그만으론 알 수 없음. 폐지 여부는 KIND 대조가 필요 | A7, A9 |
| 일일 요청 한도 | 가이드에 기간 단위 설명 없음. 파일럿에서 020이 한 번도 안 나옴 | — |
| 한 기업 안의 연결·별도 혼재 | 규칙상 발생 가능. 03번에서 건수 확인 | 사전 고정 문서 3절 |
| 정정공시 반영 여부 | 가이드 설명 없음. `rcept_no`(접수번호)를 저장하면 추적 가능 | A13 |
| 표본의 대표성 | 층화 라운드로빈이라 모집단과 비중이 다름 | A8 |

### 다음 단계
`03_추출가능성_점검.ipynb`: 받아 둔 651개 보고서에서 우리가 필요한 계정을 **실제로 찾을 수 있는지** 점검합니다.